# V024 results: small gain below the frozen promotion threshold

Both200-epoch hybrid fits completed; independent126-check audit passed. Short-window individual A MAE0.203241290; wider-window0.207317123; original curve control0.203829545.

The best fixed candidate combines the two new members and scores0.181062361 versus current0.182922095, gain0.001859734. Its breath-bootstrap interval for candidate minus reference is[-0.003404361,-0.000266644]. It falls below the predeclared0.003 gain requirement, so no F confirmation, all-data fit or submission follows this batch. Other candidates0.182502187 and0.182967119 also fail the gate. Best official mean MAE remains0.17427; target<=0.15 remains unmet.

Only approved Gym data and frozen predictions are used for analysis; no fitting/inference. New user-requested pruning will use native validation curves only, and will be implemented for future compatible A runs after this completed representative batch.

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
P=Path.cwd()
if P.name == 'research': P=P.parent
O=P/'outputs/controlled_experiments_v024'; V=O/'review'; V.mkdir(exist_ok=True)
r=json.loads((O/'screen_summary.json').read_text())
audit=json.loads((P/'research/screen_review_v024.json').read_text())
assert audit['status']=='passed' and audit['check_count']==126
assert audit['screen_summary_sha256']==hashlib.sha256((O/'screen_summary.json').read_bytes()).hexdigest()
assert r['selected_for_confirmation'] is None
rows=[dict(approach='current reference',MAE=r['reference_A_MAE'],gain=0.,eligible=False)]
rows += [{k:c[k] for k in ['name','MAE','gain','eligible']} for c in r['candidates']]
table=pd.DataFrame(rows); table['approach']=table['approach'].fillna(table['name']); table=table.drop(columns='name')
print(table.to_string(index=False))
print('\nIndividual members:')
print(pd.DataFrame(audit['members']).T[['native_A_MAE','best_epoch','runtime_seconds']].to_string())
print('\nThe best fixed gain is below the frozen0.003 promotion requirement. No extra weight search follows this batch.')

               approach      MAE      gain  eligible
      current reference 0.182922  0.000000     False
      conv3_replacement 0.182502  0.000420     False
      conv7_replacement 0.182967 -0.000045     False
conv3_conv7_replacement 0.181062  0.001860     False

Individual members:
            native_A_MAE best_epoch runtime_seconds
conv3_curve     0.203241        194     8639.920432
conv7_curve     0.207317        199     8747.707209

The best fixed gain is below the frozen0.003 promotion requirement. No extra weight search follows this batch.


In [2]:
fig,ax=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
curves={}
for name,label in [('conv3_curve','Temporal span3'),('conv7_curve','Temporal span7')]:
    events=[json.loads(x) for x in (O/name/'epochs.jsonl').read_text().splitlines()]
    history=pd.DataFrame([x for x in events if x.get('event')=='epoch']); curves[name]=history
    ax[0].plot(history.epoch,history.validation_inspiratory_mae,label=label+' validation')
    ax[1].plot(history.epoch,history.training_eval_inspiratory_mae,label=label+' training diagnostic')
old=P/'outputs/controlled_experiments_v013/deep_curve/epochs.jsonl'
events=[json.loads(x) for x in old.read_text().splitlines()]
history=pd.DataFrame([x for x in events if x.get('event')=='epoch'])
ax[0].plot(history.epoch,history.validation_inspiratory_mae,label='47-feature curve control')
ax[1].plot(history.epoch,history.training_eval_inspiratory_mae,label='47-feature curve control')
for a,title in zip(ax,['Original A validation','Unaugmented 2,048-breath training diagnostic']):
    a.set(xlabel='Epoch',ylabel='Inspiratory mean absolute error',title=title,ylim=(0,.65))
    a.grid(alpha=.2); a.legend(fontsize=8)
fig.savefig(V/'learning_curves.png',dpi=160);plt.close(fig)
print('Both fits completed200frozen epochs. Training diagnostic curves are descriptive only; future stopping decisions use validation MAE exclusively.')


Both fits completed200frozen epochs. Training diagnostic curves are descriptive only; future stopping decisions use validation MAE exclusively.


![V024 learning curves](outputs/controlled_experiments_v024/review/learning_curves.png)

In [3]:
dtype=dict(id='int64',breath_id='int32',R='int8',C='int8',u_out='int8',time_step='float32',pressure='float32')
data=pd.read_csv(P/'data/train.csv',usecols=list(dtype),dtype=dtype)
with np.load(P/'outputs/improvements_v005/sequence/split_breath_ids.npz') as z: val_ids=z['A_validation'].copy()
data=data[data.breath_id.isin(val_ids)].reset_index(drop=True)
assert len(data)==6791*80
data['step']=np.tile(np.arange(80),6791)
scored=data[data.u_out==0].reset_index(drop=True)
assert np.array_equal(scored.id.to_numpy(),np.load(O/'A_row_ids.npy'))
reference=np.load(P/'outputs/postprocessing_screen_v016/weighted_median_midpoint_then_nearest_training_pressure_codebook_A_predictions.npy')
truth=scored.pressure.to_numpy().astype(np.float64)
preds={'reference':reference}
for c in r['candidates']:preds[c['name']]=np.load(P/c['prediction_file'])
group_reports={}
for name,pred in preds.items():
    assert pred.shape==truth.shape and np.isfinite(pred).all()
    frame=scored[['breath_id','R','C','step']].copy()
    frame['absolute_error']=np.abs(pred-truth)
    for keys in [('R','C'),('step',)]:
        g=frame.groupby(list(keys)).agg(points=('absolute_error','size'),MAE=('absolute_error','mean'),total_absolute_error=('absolute_error','sum')).reset_index()
        g['fraction_total_error']=g.total_absolute_error/g.total_absolute_error.sum()
        g=g.sort_values('total_absolute_error',ascending=False)
        g.to_csv(V/(name+'_by_'+'_'.join(keys)+'.csv'),index=False)
        if keys==('R','C'):group_reports[name]=g
print('Reference lung groups, ranked by total absolute error:')
print(group_reports['reference'].to_string(index=False))
errors=np.abs(reference-truth)
breath_total=pd.Series(errors).groupby(scored.breath_id).sum().sort_values(ascending=False)
hard=breath_total.index[:3].tolist()
fig,axes=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
for ax,bid in zip(axes,hard):
    whole=data[data.breath_id==bid]; take=scored.breath_id.to_numpy()==bid
    ax.plot(whole.time_step,whole.pressure,color='black',label='Actual')
    ax.plot(scored.time_step[take],reference[take],label='Current reference')
    ax.plot(scored.time_step[take],preds['conv3_conv7_replacement'][take],label='Both hybrids',alpha=.8)
    ax.set(title=f'Breath {bid}, R={whole.R.iloc[0]}, C={whole.C.iloc[0]}',xlabel='Time (seconds)',ylabel='Pressure')
    ax.grid(alpha=.2)
axes[0].legend(fontsize=8);fig.savefig(V/'difficult_breaths.png',dpi=160);plt.close(fig)
print('\nDifficult breaths chosen by total reference absolute error:',hard)
print('Predictions shown only for scored inspiratory positions; actual pressure shown for the complete breath.')
del data

Reference lung groups, ranked by total absolute error:
 R  C  points      MAE  total_absolute_error  fraction_total_error
50 10   37444 0.188549           7060.022704              0.187269
50 20   23137 0.266569           6167.607438              0.163598
50 50   22739 0.248733           5655.948275              0.150026
 5 10   22461 0.174696           3923.844181              0.104081
20 50   21893 0.175555           3843.418287              0.101948
20 10   16589 0.179563           2978.772295              0.079013
 5 50   22061 0.124068           2737.073412              0.072602
 5 20   23070 0.116037           2676.965191              0.071007
20 20   16704 0.159017           2656.226064              0.070457

Difficult breaths chosen by total reference absolute error: [51475, 115242, 64544]
Predictions shown only for scored inspiratory positions; actual pressure shown for the complete breath.


![Difficult breaths](outputs/controlled_experiments_v024/review/difficult_breaths.png)

Lung/step groups are ranked by total absolute error. This is original A research evidence, not a new confirmation or official result.

Verified archive:83files,188843666bytes, SHA25636f77e054fe98911d6c278cb4ec6a4478b593dd70f92b7c1cac5bfef4d4d2d2a. Root source notebooks and previous files were preserved.